# 05 — Embedding Analysis

Sanity-check the embedding space — similar videos should be close, dissimilar should be far.

Includes similarity heatmaps, dimensionality reduction (t-SNE), and cluster analysis.

In [ ]:
import sys, os, json
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if "workbench" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src" / "backend"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / "workbench" / ".env")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI

print("Ready. OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "MISSING")

In [ ]:
# Load videos — either from sample-videos.json or from database
samples_path = REPO_ROOT / "workbench" / "data" / "sample-videos.json"
samples = json.loads(samples_path.read_text())

if not samples:
    print("sample-videos.json is empty. Generate test data first.")
else:
    # Build text representations for embedding
    texts = []
    labels = []
    for item in samples:
        parts = []
        if item.get("caption"): parts.append(item["caption"])
        if item.get("hashtags"): parts.append(" ".join(item["hashtags"]))
        if item.get("subtitle"): parts.append(item["subtitle"])
        text = " ".join(parts) or item.get("id", "empty")
        texts.append(text)
        labels.append(item.get("expected", {}).get("topic", "unknown"))
    
    print(f"Prepared {len(texts)} texts for embedding")
    print(f"Topic distribution: {pd.Series(labels).value_counts().to_dict()}")

In [ ]:
# Compute embeddings (batched)
def compute_embeddings_batch(texts: list[str], batch_size: int = 50) -> np.ndarray:
    """Compute embeddings using OpenAI text-embedding-3-small (same model as pipeline)."""
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model="text-embedding-3-small", input=batch)
        all_embeddings.extend([d.embedding for d in response.data])
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)}")
    return np.array(all_embeddings)

if samples and os.environ.get("OPENAI_API_KEY"):
    embeddings = compute_embeddings_batch(texts)
    print(f"Embedding matrix shape: {embeddings.shape}")
else:
    print("Skipping — need samples and OPENAI_API_KEY")

In [ ]:
# Pairwise cosine similarity heatmap
if samples and os.environ.get("OPENAI_API_KEY"):
    # Normalize for cosine similarity
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized = embeddings / norms
    similarity_matrix = normalized @ normalized.T
    
    # Plot subset (up to 50)
    n = min(50, len(similarity_matrix))
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        similarity_matrix[:n, :n],
        xticklabels=labels[:n], yticklabels=labels[:n],
        cmap="YlOrBr", vmin=0, vmax=1, ax=ax,
    )
    ax.set_title(f"Cosine Similarity Matrix ({n} videos, colored by topic)")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# t-SNE dimensionality reduction
if samples and os.environ.get("OPENAI_API_KEY") and len(embeddings) >= 5:
    from sklearn.manifold import TSNE
    
    perplexity = min(30, len(embeddings) - 1)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    coords = tsne.fit_transform(embeddings)
    
    # Plot colored by topic
    topic_colors = {t: plt.cm.tab20(i) for i, t in enumerate(sorted(set(labels)))}
    
    fig, ax = plt.subplots(figsize=(12, 8))
    for topic in sorted(set(labels)):
        mask = [l == topic for l in labels]
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            label=topic, color=topic_colors[topic], alpha=0.7, s=60,
        )
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
    ax.set_title("t-SNE Projection (colored by topic)")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 5 embedded samples for t-SNE")

In [ ]:
# Ad-hoc similarity check: embed a query and find closest videos
if samples and os.environ.get("OPENAI_API_KEY"):
    query = "funny cooking videos with recipes"
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    q_emb = np.array(client.embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding)
    
    # Cosine similarity against all videos
    sims = (normalized @ q_emb) / np.linalg.norm(q_emb)
    top_k = np.argsort(sims)[::-1][:5]
    
    print(f"Query: '{query}'")
    print(f"\nTop 5 most similar:")
    for idx in top_k:
        print(f"  [{sims[idx]:.3f}] {labels[idx]:15s} | {texts[idx][:80]}")